# Classical Baselines — WELFAKE

Adapted from **[a4-fakenews-complete-ml-ann-transformer-analysis](src/notebook/a4-fakenews-complete-ml-ann-transformer-analysis.ipynb)** (a third-party Kaggle notebook) — the actual model code (SVM, BiLSTM, CNN, Transformer encoder) is reused as-is, not reimplemented. Kept: TF-IDF+SVM, BiLSTM, CNN, and the custom Transformer encoder. Dropped vs. the source notebook: the plain ANN+TF-IDF and Embedding-ANN variants and their activation/optimizer/embedding-dim sweep cells (redundant with BiLSTM/CNN for a comparison table; those sweeps already picked embedding_dim=32, ReLU, Adam as best, which is what's used here directly).

**Two real fixes made vs. the source notebook, not carried over:**
1. The source notebook mixes an unrelated COVID-19 tweets dataset into training partway through (a tweet-vs-article confound). Every model here trains on the **same single clean split** of this one dataset instead.
2. The source notebook's final results table is **hardcoded** from a previous run, not read from the actual computed variables. Fixed here to read real values every run.

Dataset: 72,134 articles (single CSV), title+text concatenated.

**Label convention** (flipped from the source notebook to match this project's other notebooks): `0 = real`, `1 = fake`.

Also includes an **AI-generated-text-detection diagnostic** (`roberta-base-openai-detector`) — Check 1 from [../docs/verification_layer_instructions.md](../docs/verification_layer_instructions.md). WELFAKE predates the LLM era, so this is a sanity check (expect near-zero AI-generated scores on both classes) establishing that this dataset's real/fake distinction is about *source*, not *generation-origin* — a distinction the paper should be explicit about.

In [ ]:
!pip install -q transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

import warnings
warnings.filterwarnings('ignore')

# Same clean_text() used in kaggle/extract_bert_embeddings.py and
# kaggle/qhbert_end_to_end.ipynb (src/data/preprocess.py) -- the source
# notebook this was adapted from does no text cleaning at all (verified by
# an exhaustive keyword search across all 141 cells); this fixes that.
_URL_RE = re.compile(r'http\S+|www\.\S+')
_HTML_RE = re.compile(r'<.*?>')
_WHITESPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = _HTML_RE.sub(' ', str(text))
    text = _URL_RE.sub(' ', text)
    return _WHITESPACE_RE.sub(' ', text).strip()

## Data loading — WELFake

Verified real source: [saurabhshahane/fake-news-classification](https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification). Attach via + Add Input before running.

In [ ]:
import glob

def find_file(filename):
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find '{filename}' under /kaggle/input")
    return matches[0]

df = pd.read_csv(find_file('WELFake_Dataset.csv'))
df.columns = [c.strip().lower() for c in df.columns]
# source convention: 0=fake, 1=real -> flip to this project's 0=real, 1=fake
df['label'] = 1 - df['label']
df['content'] = (df['title'].astype(str) + ' ' + df['text'].astype(str)).apply(clean_text)
df = df[['content', 'label']].dropna()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df.shape)
print(df['label'].value_counts())

## Model 1 — TF-IDF + SVM

Same code as the source notebook's Model 1 (cells 26, 28).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['content'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf.shape, X_test_tfidf.shape

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)
svm_results = {
    'accuracy': accuracy_score(y_test, svm_pred),
    'precision': precision_score(y_test, svm_pred),
    'recall': recall_score(y_test, svm_pred),
    'f1': f1_score(y_test, svm_pred),
}
print('SVM:', svm_results)
print(classification_report(y_test, svm_pred))

## Shared tokenizer/padding for the deep-learning models

Same setup as the source notebook's cell 125, fit on the **same clean split** used above (not the source notebook's combined_df).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 30000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
y_train_arr = y_train.to_numpy()
y_test_arr = y_test.to_numpy()

## Model 2 — Bidirectional LSTM

Same architecture as the source notebook's Model 4 (cell 102), embedding_dim=32 (the source notebook's own sweep found this best).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

def build_bilstm_model(embedding_dim=32):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

bilstm_model = build_bilstm_model()
bilstm_model.summary()

In [ ]:
bilstm_history = bilstm_model.fit(
    X_train_pad, y_train_arr,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

bilstm_pred = bilstm_model.predict(X_test_pad).argmax(axis=1)
bilstm_results = {
    'accuracy': accuracy_score(y_test_arr, bilstm_pred),
    'precision': precision_score(y_test_arr, bilstm_pred),
    'recall': recall_score(y_test_arr, bilstm_pred),
    'f1': f1_score(y_test_arr, bilstm_pred),
}
print('BiLSTM:', bilstm_results)

## Model 3 — 1D CNN

Same architecture as the source notebook's Model 5 (cell 110).

In [ ]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

def build_cnn_model(embedding_dim=32):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
        Conv1D(filters=128, kernel_size=5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn_model()
cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    X_train_pad, y_train_arr,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1
)

In [ ]:
cnn_pred = cnn_model.predict(X_test_pad).argmax(axis=1)
cnn_results = {
    'accuracy': accuracy_score(y_test_arr, cnn_pred),
    'precision': precision_score(y_test_arr, cnn_pred),
    'recall': recall_score(y_test_arr, cnn_pred),
    'f1': f1_score(y_test_arr, cnn_pred),
}
print('CNN:', cnn_results)

## Model 4 — Transformer Encoder (from scratch)

Same architecture as the source notebook's Model 8 (cells 127-128), **trained on the same clean split as every other model here** — the source notebook trains this one on a COVID-tweets-mixed dataset instead, which is the confound this version removes.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Input, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_encoder(inputs, num_heads=4, ff_dim=128, dropout=0.1):
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim)(inputs, inputs)
    attention = Dropout(dropout)(attention)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attention)
    ffn = Dense(ff_dim, activation='relu')(out1)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(out1 + ffn)

embed_dim, num_heads, ff_dim = 64, 4, 128
inputs = Input(shape=(max_len,))
embedding_layer = Embedding(vocab_size, embed_dim)(inputs)
transformer_block = transformer_encoder(embedding_layer, num_heads, ff_dim)
x = GlobalAveragePooling1D()(transformer_block)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation='sigmoid')(x)

transformer_model = Model(inputs, outputs)
transformer_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
transformer_model.summary()

In [ ]:
transformer_history = transformer_model.fit(
    X_train_pad, y_train_arr,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

In [ ]:
transformer_pred = (transformer_model.predict(X_test_pad) > 0.5).astype('int32').flatten()
transformer_results = {
    'accuracy': accuracy_score(y_test_arr, transformer_pred),
    'precision': precision_score(y_test_arr, transformer_pred),
    'recall': recall_score(y_test_arr, transformer_pred),
    'f1': f1_score(y_test_arr, transformer_pred),
}
print('Transformer Encoder:', transformer_results)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_arr, transformer_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Transformer Encoder - Confusion Matrix')
plt.show()

## AI-generated-text detection diagnostic

Check 1 from ../docs/verification_layer_instructions.md, run as a sanity check on WELFAKE (which predates the LLM era) rather than as a classification signal — expect near-zero AI-generated scores on both classes.

In [ ]:
from transformers import pipeline

ai_text_detector = pipeline('text-classification', model='roberta-base-openai-detector', truncation=True, max_length=512)

sample_real = df[df['label'] == 0]['content'].sample(min(50, (df['label']==0).sum()), random_state=42).tolist()
sample_fake = df[df['label'] == 1]['content'].sample(min(50, (df['label']==1).sum()), random_state=42).tolist()

def mean_ai_generated_score(texts):
    scores = []
    for t in texts:
        result = ai_text_detector(t[:2000])[0]
        scores.append(result['score'] if result['label'] == 'Fake' else 1 - result['score'])
    return sum(scores) / len(scores)

ai_text_diagnostic = {
    'mean_ai_generated_score_real_class': mean_ai_generated_score(sample_real),
    'mean_ai_generated_score_fake_class': mean_ai_generated_score(sample_fake),
}
print('AI-text-detection diagnostic:', ai_text_diagnostic)

## Results summary

Pulled from the actual computed variables above (unlike the source notebook's hardcoded results dict).

In [ ]:
model_results = {
    'SVM (TF-IDF)': svm_results,
    'BiLSTM': bilstm_results,
    'CNN': cnn_results,
    'Transformer Encoder': transformer_results,
}

results_df = pd.DataFrame(model_results).T
print(results_df)

plt.figure(figsize=(10,6))
plt.plot(results_df.index, results_df['accuracy'], marker='o', linewidth=3, color='royalblue')
plt.title('Model Performance Comparison (Accuracy)')
plt.xlabel('Model'); plt.ylabel('Accuracy')
plt.ylim(max(0, results_df['accuracy'].min() - 0.05), 1.0)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Caveat — why these numbers are not the headline result

WELFAKE is a well-known 'easy' benchmark in the fake-news detection literature: real articles come from one source distribution (e.g. Reuters), fake articles from a distinctly different one, so **source/style leakage** lets almost any model — from plain TF-IDF+SVM to a from-scratch Transformer — reach ~99%. High accuracy here reflects how separable this dataset's two sources are, not genuine fake-news detection difficulty. The primary quantum-vs-classical comparison in this project's paper should center on **LIAR** (harder, and the dataset HQDNN also reports numbers on in docs/qhbert_papers_detailed_reference.md) — these numbers are supporting breadth, not the headline result.

In [ ]:
import json

summary = {'dataset': 'welfake', 'baselines': model_results, 'ai_text_detection': ai_text_diagnostic}
with open('/kaggle/working/welfake_classical_baselines.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))